# Exercício 2 — Simulação de Consultório Médico

## Descrição do Sistema

Um consultório médico especializado em clínica geral atende pacientes ao longo do dia utilizando um sistema de agendamento combinado com encaixes de emergência. O consultório opera com disciplina de fila **FIFO** e possui os seguintes recursos:

- 1 recepcionista
- 1 sala de triagem
- 2 médicos

O fluxo do paciente segue a sequência: cadastro na recepção $\rightarrow$ triagem $\rightarrow$ aguarda atendimento médico $\rightarrow$ consulta $\rightarrow$ saída do sistema.

---

## a) Identificação dos Componentes do Sistema

### Entidades
- Pacientes (agendados e encaixes de emergência)

### Recursos
- 1 recepcionista (responsável pelo cadastro)
- 1 sala de triagem
- 2 médicos (atendimento simultâneo possível)

### Filas
- Fila da recepção: aguarda início do cadastro
- Fila da triagem: aguarda início da triagem
- Fila médica: aguarda início da consulta

### Eventos
- Chegada do paciente ao sistema
- Início e fim do cadastro
- Início e fim da triagem
- Início e fim da consulta médica
- Saída do sistema

### Variáveis de Estado
- Número de pacientes em cada fila (recepção, triagem, médica)
- Estado da recepcionista: livre / ocupada
- Estado da sala de triagem: livre / ocupada
- Estado de cada médico: livre / ocupado (2 servidores independentes)

---

## b) Diagrama ACD (Atividades, Filas e Decisões)

*(O diagrama visual se encontra no relatório principal em formato PDF)*

---

## c) Fases de Simulação (Abordagem de Três Fases)

### Fase A — Avanço do Relógio
O relógio de simulação avança para o instante do próximo evento programado na lista de eventos futuros (FEL — *Future Event List*). Os eventos considerados são: chegadas, fins de cadastro, fins de triagem e fins de consulta.

### Fase B — Eventos Incondicionais
Ocorrem independentemente do estado do sistema:
- **Chegada:** gera a próxima chegada e insere o paciente na fila da recepção.
- **Fim do cadastro:** libera a recepcionista e move o paciente para a fila de triagem.
- **Fim da triagem:** libera a sala de triagem e move o paciente para a fila médica.
- **Fim da consulta:** libera o médico e registra a saída do paciente.

### Fase C — Eventos Condicionais
Dependem da disponibilidade de recursos:
- **Início do cadastro:** fila da recepção não vazia **e** recepcionista livre.
- **Início da triagem:** fila de triagem não vazia **e** sala de triagem livre.
- **Início da consulta:** fila médica não vazia **e** pelo menos um médico livre.

---

## d) Dinâmica da Simulação

A dinâmica deste sistema baseia-se no método de **Simulação de Três Fases**, que garante que o estado do consultório seja atualizado de forma cronológica e logicamente consistente. A argumentação abaixo detalha como esses processos interagem:

- **Atualização do Relógio de Simulação (Avanço por Evento):** Diferente de uma simulação por intervalos de tempo fixos, este modelo utiliza o *Next-Event Time Advance*. O relógio "salta" diretamente para o instante do evento mais iminente na *Future Event List* (FEL). Esta abordagem é computacionalmente eficiente e garante que nenhum evento intermediário seja perdido, tratando o tempo como uma variável discreta.
- **Ativação de Servidores e Verificação de Recursos:** A ativação dos servidores (recepcionista, sala de triagem e médicos) é regida por uma lógica de disponibilidade. Quando um evento de "Fim de Atividade" ocorre, o servidor não apenas se torna livre, mas dispara imediatamente uma verificação na fila correspondente. Se houver entidades aguardando, o início de uma nova atividade é agendado (Fase C).
- **Movimentação entre Filas e Sincronismo:** A movimentação do paciente através do consultório é um processo de "empurrar e puxar" (*push-pull*). Ao concluir uma etapa, o paciente é "empurrado" para a próxima fila. Se o recurso subsequente estiver livre, o paciente é "puxado" para o atendimento. Este sincronismo é crítico no modelo, especialmente na transição da triagem para a consulta médica.
- **Verificação de Recursos Livres (Fase C):** A cada avanço do relógio, o sistema reavalia todas as condições de bloqueio. Um evento condicional só é disparado se a conjunção lógica (*Fila não vazia* $\land$ *Recurso disponível*) for verdadeira. No caso dos médicos, a regra de decisão garante o processamento paralelo que caracteriza este sistema multiserver.

---

## e) Tabela de Simulação Manual

Dados fornecidos (Tempos de serviço por paciente):

| | P1 | P2 | P3 | P4 | P5 |
|---|---|---|---|---|---|
| **Entre chegadas (min)** | 5 | 7 | 3 | 10 | 6 |
| **Cadastro (min)** | 4 | 5 | 3 | 4 | 6 |
| **Triagem (min)** | 6 | 5 | 7 | 4 | 6 |
| **Consulta (min)** | 12 | 18 | 10 | 20 | 15 |

**Tabela de simulação manual — tempos em minutos:**

| Paciente | Chegada | Início Cadastro | Fim Cadastro | Início Triagem | Fim Triagem | Início Consulta | Fim Consulta |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| **P1** | 5  | 5  | 9  | 9  | 15 | 15 | 27 |
| **P2** | 12 | 12 | 17 | 17 | 22 | 22 | 40 |
| **P3** | 15 | 17 | 20 | 22 | 29 | 29 | 39 |
| **P4** | 25 | 25 | 29 | 29 | 33 | 39 | 59 |
| **P5** | 31 | 31 | 37 | 37 | 43 | 43 | 58 |

> *Nota: Os 2 médicos permitem atendimentos paralelos. P1 ocupa o médico 1 a partir de $t=15$ e P2 ocupa o médico 2 a partir de $t=22$. P3 inicia consulta no médico 1 em $t=29$, assim que este fica livre.*

---

## f) Cálculo das Métricas e Discussão dos Resultados

### 1. Tempo Médio na Fila da Recepção
$$W_{fila\_rec} = \frac{(5-5)+(12-12)+(17-15)+(25-25)+(31-31)}{5} = \frac{0+0+2+0+0}{5} = \mathbf{0{,}4 \text{ min}}$$

### 2. Tempo Médio na Fila Médica
$$W_{fila\_med} = \frac{(15-15)+(22-22)+(29-29)+(39-33)+(43-43)}{5} = \frac{0+0+0+6+0}{5} = \mathbf{1{,}2 \text{ min}}$$

### 3. Tempo Médio Total no Sistema
$$W_{total} = \frac{(27-5)+(40-12)+(39-15)+(59-25)+(58-31)}{5} = \frac{22+28+24+34+27}{5} = \mathbf{27 \text{ min}}$$

### 4. Taxa de Utilização dos Médicos
Considerando o horizonte de simulação até a saída do último paciente ($T = 59$ min):
$$\rho_{médicos} = \frac{\text{tempo total de consultas}}{n_{médicos} \times T_{simulação}} = \frac{12+18+10+20+15}{2 \times 59} = \frac{75}{118} \approx \mathbf{63{,}56\%}$$

### 5. Número Médio de Pacientes no Sistema
Pelo Teorema de Little ($L = \lambda W$), utilizando a taxa de chegada média sobre o tempo total:
$$\lambda = \frac{5 \text{ pacientes}}{59 \text{ min}} \approx 0{,}0847 \text{ pac/min}$$
$$L = \lambda \times W_{total} = 0{,}0847 \times 27 \approx \mathbf{2{,}29 \text{ pacientes}}$$

### Análise dos Resultados
- **Eficiência da Recepção:** O tempo médio de espera de 0,4 min indica que o recurso de recepção é adequado. O pequeno gargalo observado no P3 (espera de 2 min) é rapidamente solucionado.
- **Dinâmica da Fila Médica:** A fila médica apresentou a maior espera (pico de 6 min para o P4). Apesar de haver dois médicos, a alta variabilidade nos tempos de consulta (como os 18 min do P2) gera pequenas retenções temporárias.
- **Ocupação e Dimensionamento:** A taxa de utilização de $\approx 63,5\%$ revela um sistema saudável e equilibrado. Ademais, o número médio de $\approx 2,29$ pacientes no sistema indica que as acomodações (sala de espera) não sofrerão superlotação com a atual taxa de chegadas.